## 🔹 [Step 3] 최신 OpenAI Responses API 기반 챗봇 백엔드 로직 (`BaseballBotAPI`)

OpenAI의 최신 **Responses API (`client.responses.create`)**를 적용한 백엔드 API 클래스입니다.

### 💡 기존 Chat Completions 대비 최신 Responses API의 장점
1. **`instructions` 파라미터**: 시스템 프롬프트(페르소나/지침)를 일반 대화와 명확히 분리하여 전달
2. **직관적인 `response.output_text`**: 기존의 복잡한 `response.choices[0].message.content` 대신 단일 속성으로 텍스트 바로 획득
3. **멀티턴 대화 (`input`)**: 이전 대화 히스토리 리스트(`[{'role': '...', 'content': '...'}]`)를 `input`으로 간결하게 전달
4. **향후 웹 검색 및 도구 확장성**: 향후 뉴스 실시간 검색, 함수 호출(도구) 연동 시에도 동일한 통일된 인터페이스 제공

## 🔹 [Step 1] 환경 설정 및 OpenAI API 연동 준비

`.env` 파일에 설정된 `OPENAI_API_KEY`를 로드하여 OpenAI 클라이언트를 초기화합니다.
또한 UI 구동에 필요한 `pywebview` 라이브러리의 정상 작동 여부를 점검합니다.

In [7]:
import os
import json
import time
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import webview
from importlib.metadata import version

# 1. .env 파일 환경 변수 로드
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("❌ .env 파일에서 OPENAI_API_KEY를 찾을 수 없습니다. 키를 확인해주세요.")

# 2. OpenAI 클라이언트 인스턴스 생성
openai_client = OpenAI(api_key=api_key)

print("✅ 환경 설정 완료!")
print(f"- OpenAI API Key: {api_key[:7]}...{api_key[-4:]} (정상 로드됨)")
print("pywebview가 정상적으로 로드되었습니다.")


✅ 환경 설정 완료!
- OpenAI API Key: sk-proj...n98A (정상 로드됨)
pywebview가 정상적으로 로드되었습니다.


## 🔹 [Step 2] 챗봇 프론트엔드 UI 구축 (HTML / CSS / JavaScript)

야구 테마의 직관적인 디자인과 부드러운 반응성을 갖춘 챗봇 화면입니다.
- **주요 구성**: 상단 네이비 헤더, 채팅 말풍선 영역, 실시간 타이핑 로딩 애니메이션, 원클릭 퀵 질문 버튼, 메시지 입력 폼
- **pywebview 브릿지**: `window.pywebview.api.send_message()`를 통해 백엔드 파이썬과 비동기로 통신합니다.

In [8]:
# 프론트엔드 UI 템플릿 코드 정의
HTML_CODE = '''<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>야구 뉴스 브리핑 AI</title>
    <style>
        :root {
            --primary: #0f2b5c;
            --primary-light: #1e3d7a;
            --accent: #e63946;
            --accent-hover: #d62839;
            --bg: #f4f6f9;
            --card-bg: #ffffff;
            --text-dark: #1e293b;
            --text-muted: #64748b;
            --border: #e2e8f0;
            --bot-bubble: #ffffff;
            --user-bubble: #0f2b5c;
        }

        * {
            box-sizing: border-box;
            margin: 0;
            padding: 0;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Pretendard', 'Malgun Gothic', sans-serif;
        }

        body {
            background: var(--bg);
            height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            overflow: hidden;
        }

        .chat-container {
            width: 100%;
            height: 100vh;
            background: var(--card-bg);
            display: flex;
            flex-direction: column;
            box-shadow: 0 8px 30px rgba(0, 0, 0, 0.08);
        }

        /* Header */
        .chat-header {
            background: linear-gradient(135deg, var(--primary) 0%, var(--primary-light) 100%);
            color: white;
            padding: 18px 24px;
            display: flex;
            align-items: center;
            justify-content: space-between;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }

        .header-title-wrap {
            display: flex;
            align-items: center;
            gap: 12px;
        }

        .logo-badge {
            font-size: 26px;
            background: rgba(255, 255, 255, 0.15);
            width: 44px;
            height: 44px;
            display: flex;
            align-items: center;
            justify-content: center;
            border-radius: 12px;
            border: 1px solid rgba(255, 255, 255, 0.25);
        }

        .header-text h1 {
            font-size: 1.15rem;
            font-weight: 700;
            letter-spacing: -0.3px;
        }

        .header-text p {
            font-size: 0.8rem;
            color: rgba(255, 255, 255, 0.8);
            margin-top: 2px;
        }

        .status-badge {
            display: inline-flex;
            align-items: center;
            gap: 6px;
            background: rgba(16, 185, 129, 0.2);
            border: 1px solid rgba(16, 185, 129, 0.4);
            color: #34d399;
            font-size: 0.75rem;
            padding: 4px 10px;
            border-radius: 20px;
            font-weight: 600;
        }

        .status-dot {
            width: 7px;
            height: 7px;
            background: #10b981;
            border-radius: 50%;
            box-shadow: 0 0 8px #10b981;
        }

        /* Message Area */
        .chat-messages {
            flex: 1;
            padding: 24px;
            overflow-y: auto;
            display: flex;
            flex-direction: column;
            gap: 18px;
            background: #f8fafc;
        }

        .message-row {
            display: flex;
            align-items: flex-start;
            gap: 10px;
            max-width: 82%;
            animation: fadeIn 0.25s ease-out;
        }

        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(6px); }
            to { opacity: 1; transform: translateY(0); }
        }

        .message-row.user {
            align-self: flex-end;
            flex-direction: row-reverse;
        }

        .avatar {
            width: 36px;
            height: 36px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 18px;
            flex-shrink: 0;
        }

        .avatar.bot {
            background: #e2e8f0;
            border: 1px solid #cbd5e1;
        }

        .avatar.user {
            background: #dbeafe;
            border: 1px solid #bfdbfe;
        }

        .bubble {
            padding: 13px 18px;
            border-radius: 16px;
            font-size: 0.93rem;
            line-height: 1.55;
            word-break: break-word;
            box-shadow: 0 1px 3px rgba(0,0,0,0.05);
            position: relative;
        }

        .message-row.bot .bubble {
            background: var(--bot-bubble);
            color: var(--text-dark);
            border: 1px solid var(--border);
            border-top-left-radius: 4px;
        }

        .message-row.user .bubble {
            background: var(--user-bubble);
            color: #ffffff;
            border-top-right-radius: 4px;
        }

        .timestamp {
            font-size: 0.7rem;
            color: var(--text-muted);
            margin-top: 4px;
            display: block;
        }

        .message-row.user .timestamp {
            text-align: right;
            color: rgba(255,255,255,0.7);
        }

        /* Quick Buttons */
        .quick-actions {
            padding: 8px 24px;
            background: #f8fafc;
            display: flex;
            gap: 8px;
            overflow-x: auto;
            border-top: 1px solid #f1f5f9;
        }

        .quick-btn {
            background: #ffffff;
            border: 1px solid #cbd5e1;
            border-radius: 20px;
            padding: 6px 14px;
            font-size: 0.82rem;
            color: var(--text-dark);
            cursor: pointer;
            white-space: nowrap;
            transition: all 0.2s ease;
            display: inline-flex;
            align-items: center;
            gap: 4px;
        }

        .quick-btn:hover {
            background: #eff6ff;
            border-color: #93c5fd;
            color: var(--primary);
            transform: translateY(-1px);
        }

        /* Typing Indicator */
        .typing-indicator {
            display: none;
            align-items: center;
            gap: 4px;
            padding: 12px 18px;
            background: white;
            border: 1px solid var(--border);
            border-radius: 16px;
            border-top-left-radius: 4px;
            width: fit-content;
        }

        .typing-dot {
            width: 7px;
            height: 7px;
            background: #94a3b8;
            border-radius: 50%;
            animation: bounce 1.4s infinite ease-in-out both;
        }

        .typing-dot:nth-child(1) { animation-delay: -0.32s; }
        .typing-dot:nth-child(2) { animation-delay: -0.16s; }

        @keyframes bounce {
            0%, 80%, 100% { transform: scale(0); }
            40% { transform: scale(1); }
        }

        /* Input Area */
        .chat-input-container {
            padding: 16px 24px 20px;
            background: white;
            border-top: 1px solid var(--border);
        }

        .input-form {
            display: flex;
            gap: 10px;
            align-items: center;
        }

        .chat-input {
            flex: 1;
            padding: 13px 18px;
            border: 1.5px solid var(--border);
            border-radius: 24px;
            font-size: 0.95rem;
            outline: none;
            transition: border-color 0.2s ease, box-shadow 0.2s ease;
        }

        .chat-input:focus {
            border-color: var(--primary-light);
            box-shadow: 0 0 0 3px rgba(30, 61, 122, 0.12);
        }

        .send-button {
            background: var(--accent);
            color: white;
            border: none;
            width: 46px;
            height: 46px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            transition: all 0.2s ease;
            box-shadow: 0 3px 8px rgba(230, 57, 70, 0.3);
        }

        .send-button:hover {
            background: var(--accent-hover);
            transform: translateY(-1px);
            box-shadow: 0 4px 12px rgba(230, 57, 70, 0.4);
        }

        .send-button:active {
            transform: translateY(1px);
        }

        .send-button svg {
            width: 20px;
            height: 20px;
            fill: white;
        }
    </style>
</head>
<body>
    <div class="chat-container">
        <!-- 헤더 -->
        <header class="chat-header">
            <div class="header-title-wrap">
                <div class="logo-badge">⚾</div>
                <div class="header-text">
                    <h1>야구 뉴스 브리핑 AI</h1>
                    <p>OpenAI 연동 · 기사 요약 및 데이터 저장 챗봇</p>
                </div>
            </div>
            <div class="status-badge">
                <span class="status-dot"></span>
                <span>AI 연결됨</span>
            </div>
        </header>

        <!-- 채팅 메시지 영역 -->
        <div class="chat-messages" id="chatMessages">
            <div class="message-row bot">
                <div class="avatar bot">⚾</div>
                <div>
                    <div class="bubble">
                        안녕하세요! <strong>야구 뉴스 브리핑 AI</strong>입니다. ⚾<br><br>
                        OpenAI API가 연결되어 실시간 질의응답이 가능합니다.<br>
                        KBO 리그, 구단 소식, 관심 있는 선수에 대해 무엇이든 질문하시거나 아래 퀵 버튼을 눌러보세요!
                        <span class="timestamp">시스템 시작</span>
                    </div>
                </div>
            </div>

            <!-- 로딩 인디케이터 -->
            <div class="message-row bot typing-indicator" id="typingIndicator">
                <div class="avatar bot">⚾</div>
                <div class="bubble" style="display:flex; align-items:center; gap:5px; padding: 12px 16px;">
                    <span class="typing-dot"></span>
                    <span class="typing-dot"></span>
                    <span class="typing-dot"></span>
                </div>
            </div>
        </div>

        <!-- 퀵 추천 질문 버튼 -->
        <div class="quick-actions">
            <button class="quick-btn" onclick="sendQuickMessage('최근 KBO 리그 주요 관전 포인트 알려줘')">⚾ 관전 포인트</button>
            <button class="quick-btn" onclick="sendQuickMessage('야구 기사 스크랩 및 요약 보고서는 어떤 방식으로 진행돼?')">📰 기사 수집 안내</button>
            <button class="quick-btn" onclick="sendQuickMessage('수집한 야구 데이터를 CSV 파일로 저장하는 방법 설명해줘')">💾 CSV 저장 안내</button>
        </div>

        <!-- 입력창 -->
        <div class="chat-input-container">
            <form class="input-form" id="chatForm" onsubmit="handleSubmit(event)">
                <input type="text" id="userInput" class="chat-input" placeholder="야구에 대해 자유롭게 질문해보세요..." autocomplete="off" autofocus>
                <button type="submit" class="send-button" title="전송">
                    <svg viewBox="0 0 24 24">
                        <path d="M2.01 21L23 12 2.01 3 2 10l15 2-15 2z"/>
                    </svg>
                </button>
            </form>
        </div>
    </div>

    <script>
        const messagesContainer = document.getElementById('chatMessages');
        const userInput = document.getElementById('userInput');
        const typingIndicator = document.getElementById('typingIndicator');

        function getTimeString() {
            const now = new Date();
            return now.toLocaleTimeString('ko-KR', { hour: '2-digit', minute: '2-digit' });
        }

        function scrollToBottom() {
            messagesContainer.scrollTop = messagesContainer.scrollHeight;
        }

        function appendMessage(sender, text) {
            const row = document.createElement('div');
            row.className = `message-row ${sender}`;

            const avatar = document.createElement('div');
            avatar.className = `avatar ${sender}`;
            avatar.innerText = sender === 'bot' ? '⚾' : '👤';

            const bubbleWrap = document.createElement('div');
            const bubble = document.createElement('div');
            bubble.className = 'bubble';
            
            // 줄바꿈 및 볼드 마크다운 기본 포맷팅
            let formatted = text
                .replace(/\n/g, '<br>')
                .replace(/\*\*(.*?)\*\*/g, '<strong>$1</strong>');
            
            bubble.innerHTML = formatted + `<span class="timestamp">${getTimeString()}</span>`;
            
            bubbleWrap.appendChild(bubble);
            row.appendChild(avatar);
            row.appendChild(bubbleWrap);

            messagesContainer.insertBefore(row, typingIndicator);
            scrollToBottom();
        }

        function showTyping(show) {
            typingIndicator.style.display = show ? 'flex' : 'none';
            if (show) scrollToBottom();
        }

        async function handleSubmit(e) {
            if (e) e.preventDefault();
            const text = userInput.value.trim();
            if (!text) return;

            appendMessage('user', text);
            userInput.value = '';
            showTyping(true);

            try {
                if (window.pywebview && window.pywebview.api) {
                    const response = await window.pywebview.api.send_message(text);
                    showTyping(false);
                    if (response && response.reply) {
                        appendMessage('bot', response.reply);
                    } else {
                        appendMessage('bot', '응답을 가져오지 못했습니다.');
                    }
                } else {
                    setTimeout(() => {
                        showTyping(false);
                        appendMessage('bot', `[브라우저 테스트 모드]\n'${text}'\npywebview 환경에서 파이썬 백엔드(OpenAI)와 실시간 연동됩니다.`);
                    }, 600);
                }
            } catch (err) {
                showTyping(false);
                appendMessage('bot', `오류가 발생했습니다: ${err.message || err}`);
            }
        }

        function sendQuickMessage(text) {
            userInput.value = text;
            handleSubmit();
        }

        window.addEventListener('pywebviewready', () => {
            console.log('pywebview 브릿지가 준비되었습니다.');
        });
    </script>
</body>
</html>'''
print("✅ 프론트엔드 UI 템플릿 준비 완료!")

✅ 프론트엔드 UI 템플릿 준비 완료!


<>:407: SyntaxWarning: invalid escape sequence '\*'
<>:407: SyntaxWarning: invalid escape sequence '\*'
C:\Users\Admin\AppData\Local\Temp\ipykernel_3564\3868550569.py:407: SyntaxWarning: invalid escape sequence '\*'
  .replace(/\*\*(.*?)\*\*/g, '<strong>$1</strong>');


## 🔹 [Step 3] 최신 OpenAI Responses API 기반 챗봇 백엔드 로직 (`BaseballBotAPI`)

OpenAI의 최신 **Responses API (`client.responses.create`)**를 적용한 백엔드 API 클래스입니다.

### 💡 기존 Chat Completions 대비 최신 Responses API의 장점
1. **`instructions` 파라미터**: 시스템 프롬프트(페르소나/지침)를 일반 대화와 명확히 분리하여 전달
2. **직관적인 `response.output_text`**: 기존의 복잡한 `response.choices[0].message.content` 대신 단일 속성으로 텍스트 바로 획득
3. **멀티턴 대화 (`input`)**: 이전 대화 히스토리 리스트(`[{'role': '...', 'content': '...'}]`)를 `input`으로 간결하게 전달
4. **향후 웹 검색 및 도구 확장성**: 향후 뉴스 실시간 검색, 함수 호출(도구) 연동 시에도 동일한 통일된 인터페이스 제공

In [ ]:
class BaseballBotAPI:
    """
    최신 OpenAI Responses API(client.responses.create)를 활용한 야구 뉴스 챗봇 백엔드 API
    """
    def __init__(self, client: OpenAI, model: str = "gpt-5.6-luna"):
        self.client = client
        self.model = model
        
        # 최신 API의 instructions: 시스템 페르소나 및 동작 지침
        self.instructions = (
            "당신은 한국 프로야구(KBO) 및 야구 뉴스 전문 AI 어시스턴트 '야구 브리핑 AI'입니다.\n"
            "사용자에게 친절하고 신뢰감 있는 어조로 답변하세요.\n"
            "[주요 역할 및 안내]\n"
            "1. 야구 경기 규칙, 구단 소식, 최신 이슈 및 선수 정보에 대해 명확하고 흥미롭게 답변합니다.\n"
            "2. 향후 단계에서 네이버 스포츠 등 실시간 야구 뉴스 기사 크롤링, 기사 요약 리포트 생성, "
            "스크랩 데이터를 CSV 파일로 저장하는 기능을 순차적으로 제공할 예정임을 안내합니다.\n"
            "3. 답변은 가독성 좋게 핵심 위주로 불릿포인트나 단락을 나누어 작성하세요."
        )
        
        # 대화 기록 유지 (멀티턴 지원)
        self.conversation_history = []

    def send_message(self, message: str) -> dict:
        """
        프론트엔드 JavaScript에서 호출하는 메시지 처리 함수
        """
        user_text = message.strip()
        if not user_text:
            return {"status": "error", "reply": "질문 내용을 입력해주세요."}

        # 1. 사용자 질문을 대화 기록에 추가
        self.conversation_history.append({"role": "user", "content": user_text})

        # 최근 10개 대화(5턴)를 input으로 전달하여 맥락 유지
        trimmed_input = self.conversation_history[-10:]

        try:
            # 2. 최신 OpenAI Responses API 호출
            response = self.client.responses.create(
                model=self.model,
                instructions=self.instructions,
                input=trimmed_input
            )
            
            # 3. 최신 응답 텍스트 추출 (response.output_text)
            bot_reply = response.output_text.strip()
            
            # 4. AI 답변을 대화 기록에 저장
            self.conversation_history.append({"role": "assistant", "content": bot_reply})
            
            return {"status": "success", "reply": bot_reply}

        except Exception as e:
            error_msg = f"⚠️ OpenAI API 호출 중 오류가 발생했습니다: {str(e)}"
            print(error_msg)
            return {"status": "error", "reply": error_msg}

# API 인스턴스 초기화 및 최신 API 응답 테스트
bot_api = BaseballBotAPI(client=openai_client)
test_response = bot_api.send_message("야구 뉴스 브리핑 AI 소개를 한 줄로 해줘!")
print("\n[최신 OpenAI Responses API 응답 테스트]")
print(test_response["reply"])


[OpenAI API 응답 테스트]
한국 프로야구(KBO) 및 야구 관련 최신 소식과 규칙 정보를 친절하게 제공하는 AI 어시스턴트입니다!


## 🔹 [Step 4] `pywebview` 데스크톱 챗봇 윈도우 실행

앞서 정의한 HTML UI와 OpenAI 연동 `bot_api`를 결합하여 데스크톱 챗봇 윈도우를 실행합니다.
- 주피터 노트북에서 아래 셀을 실행하면 데스크톱 팝업 창이 열립니다.
- 창 내에서 메시지를 입력하면 실제 OpenAI API를 통해 실시간 야구 브리핑 답변을 받아볼 수 있습니다.

In [10]:
def start_chatbot():
    """
    야구 뉴스 브리핑 AI 데스크톱 웹뷰 창을 생성하고 실행합니다.
    """
    # 1. 윈도우 생성 (제목, HTML 코드, JS API 바인딩, 창 크기)
    window = webview.create_window(
        title="⚾ 야구 뉴스 브리핑 AI 챗봇",
        html=HTML_CODE,
        js_api=bot_api,
        width=920,
        height=720,
        resizable=True,
        min_size=(640, 480)
    )
    
    # 2. 웹뷰 창 시작 (debug=True로 우클릭 검사/F12 개발자도구 지원)
    print("🚀 데스크톱 챗봇 창을 실행합니다. (창을 닫으면 셀 실행이 완료됩니다)")
    webview.start(debug=True)

# 아래 함수를 호출하여 챗봇 실행
start_chatbot()

🚀 데스크톱 챗봇 창을 실행합니다. (창을 닫으면 셀 실행이 완료됩니다)


[pywebview] before_load event fired. injecting pywebview object
[pywebview] Loading JS files from C:\Projects\mini-project_0909\.venv\Lib\site-packages\webview\js
[pywebview] _pywebviewready event fired
[pywebview] loaded event fired


## 🔹 [Step 5 & 6 (예정)] 향후 추가 개발 단계 안내

기본 챗봇과 OpenAI API 연동이 완료되었으므로, 다음 단계에서 순차적으로 기능을 확장할 예정입니다:

### 1. 야구 뉴스 기사 수집 모듈 (`beautifulsoup4`, `selenium`)
- 네이버 스포츠 야구 뉴스 최신 기사 크롤링
- 기사 제목, 링크, 언론사, 발행일자, 기사 본문 수집

### 2. 기사 요약 및 리포트 작성 (`openai`)
- 수집된 뉴스 기사를 AI가 3줄 핵심 요약 및 시사점 도출

### 3. 데이터프레임 변환 및 CSV 저장 (`pandas`)
- 스크랩된 기사 데이터를 Pandas DataFrame으로 정돈 후 `baseball_news_YYYYMMDD.csv` 저장
- 챗봇 UI에서 'CSV 저장하기' 버튼 클릭 시 파일 자동 생성 기능 연동